# Calculate the amplitudes

This notebook is used to calculate the amplitudes of the seismograms.

Agentic AI was used in this notebook.

By Hiroto Bito

In [ ]:
# Import necessary libraries
import os
import sys
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

from obspy import read, UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.stream import Stream

parent_dir = '/home/hbito/cascadia_obs_ensemble/utils'
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from data_client import get_waveforms

In [ ]:
# Read the data frame
datasets_dir =  '/wd1/hbito_data/data/datasets_all_regions'
path_assigned_picks_df = f'{datasets_dir}/Cascadia_updated_catalog_picks_assignment_ver_3.csv'

# Prepare output CSV path 
output_csv_path = f'{datasets_dir}/Cascadia_updated_catalog_picks_assignment_ver_3_w_amp.csv'

# File to save skipped picks
skipped_csv_path = f'{datasets_dir}/calculate_amplitudes_skipped_picks.csv'

assigned_picks_df = pd.read_csv(path_assigned_picks_df, index_col=False).copy()

,Unnamed: 0,time_pick,station,phase,timeres,idx,arid,latitude,longitude,depth,Detection Value,time,RMS Residual (s),Num. P,Num. S,picks,slatitude,slongitude,selevation
0,0,2010-01-01T00:15:27.180000Z,UW.PCMD,P,0.049,0,0,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
1,1,2010-01-01T00:15:37.840400Z,UW.RVW,P,1.264,0,1,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.149750,-122.742996,504.0
2,2,2010-01-01T00:15:33.280000Z,UW.PCMD,S,-0.243,0,2,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
3,3,2010-01-01T00:15:42.002000Z,UW.GNW,S,2.402,0,3,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.564130,-122.824980,220.0
4,4,2010-01-01T00:15:43.618400Z,PB.B013,S,-0.651,0,4,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813000,-122.910797,75.3
5,5,2010-01-01T00:15:43.768400Z,PB.B943,S,-0.511,0,5,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813202,-122.911301,84.2
6,6,2010-01-01T00:15:48.060400Z,UW.BOW,S,-0.263,0,6,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.474831,-123.229301,870.0
7,7,2010-01-01T00:17:04.730000Z,UW.PASS,P,-0.499,1,7,48.17742,-121.83289,6.163,0.84,2010-01-01 00:16:49.343000+00:00,0.985,25,30,55,48.998299,-122.085197,175.4
8,8,2010-01-01T00:17:05.008400Z,PB.B943,P,-0.252,1,8,48.17742,-121.83289,6.163,0.84,2010-01-01 00:16:49.343000+00:00,0.985,25,30,55,47.813202,-122.911301,84.2
9,9,2010-01-01T00:17:05.020400Z,UW.BLN,P,0.415,1,9,48.17742,-121.83289,6.163,0.84,2010-01-01 00:16:49.343000+00:00,0.985,25,30,55,48.006624,-122.972646,601.0


test

In [7]:
assigned_picks_df

,Unnamed: 0,time_pick,station,phase,timeres,idx,arid,latitude,longitude,depth,Detection Value,time,RMS Residual (s),Num. P,Num. S,picks,slatitude,slongitude,selevation
0,0,2010-01-01T00:15:27.180000Z,UW.PCMD,P,0.049,0,0,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
1,1,2010-01-01T00:15:37.840400Z,UW.RVW,P,1.264,0,1,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.149750,-122.742996,504.0
2,2,2010-01-01T00:15:33.280000Z,UW.PCMD,S,-0.243,0,2,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
3,3,2010-01-01T00:15:42.002000Z,UW.GNW,S,2.402,0,3,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.564130,-122.824980,220.0
4,4,2010-01-01T00:15:43.618400Z,PB.B013,S,-0.651,0,4,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813000,-122.910797,75.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1004330,1004330,2015-06-23T23:18:40.885798Z,7D.J11D,S,0.044,63886,1004330,43.33308,-127.31240,6.0615,0.694,2015-06-23 23:18:19.457000+00:00,0.447,4,5,9,43.541599,-126.368599,-3000.8
1004331,1004331,2015-06-23T23:18:48.573898Z,7D.G35D,S,0.358,63886,1004331,43.33308,-127.31240,6.0615,0.694,2015-06-23 23:18:19.457000+00:00,0.447,4,5,9,42.555698,-126.399002,-2822.6
1004332,1004332,2015-06-23T23:18:50.458298Z,7D.J19D,S,0.300,63886,1004332,43.33308,-127.31240,6.0615,0.694,2015-06-23 23:18:19.457000+00:00,0.447,4,5,9,44.179001,-126.271202,-2955.4
1004333,1004333,2015-06-23T23:18:56.689277Z,7D.J10D,S,0.432,63886,1004333,43.33308,-127.31240,6.0615,0.694,2015-06-23 23:18:19.457000+00:00,0.447,4,5,9,43.348499,-125.545097,-3085.0


end test

In [1]:
# Randomly pick a subset of the table
n_picks = 500
subset = assigned_picks_df.iloc[:n_picks].copy()
subset

NameError: name 'assigned_picks_df' is not defined

In [4]:
# Define the arguments
window_before = 0.5 # in sec
window_after = 2 # in sec
source = 'pnwstore'

freq_highpass = 2 # in Hz
new_sampling_rate = 100 # in Hz

In [5]:
# Run the loop
amplitudes = []

for idx, row in subset.iterrows():

    # Define the arguments 
    date, _time = row['time'].split(' ')
    datetime_str = date+'T'+_time
    origin_time = UTCDateTime(datetime_str)  # Accept ISO string directly

    network = row['station'].split('.')[0].strip()
    station = row['station'].split('.')[1].strip()
    channel = '*H*'
    starttime = origin_time - window_before 
    endtime = origin_time + window_after

    time_pick = row['time_pick']        

    # Request a waveform
    time.sleep(0.1)

    try:
        st = get_waveforms(network=network, station=station, channel=channel,
                            starttime=starttime, endtime=endtime,
                            source=source)
    except Exception as e:
        print(f"Request failed: {e}")
        st = Stream()

    time.sleep(0.1)


    # Create a new stream
    sdata = Stream()
    
    # Check if loaded data have a vertical component (minimum requirement)
    has_Z = bool(st.select(id=f'{network}.{station}..??Z'))
    # Check for HH and BH channels presence
    has_HH = bool(st.select(id=f'{network}.{station}..HH?'))
    has_BH = bool(st.select(id=f'{network}.{station}..BH?'))
    has_EH = bool(st.select(id=f'{network}.{station}..EH?'))

    if not has_Z:
        print(f'No Vertical Component Data Present at {network}.{station} with HHZ, BHZ or EHZ channels at {time_pick}. Skipping')
        continue

    # Apply selection logic based on channel presence
    if has_HH:
        # If all HH, BH, and EH, channels are present, select only HH
        sdata += st.select(id=f'{network}.{station}..HH*')
    elif has_BH:
        # If BH and EH channels are present, select only BH
        sdata += st.select(id=f'{network}.{station}..BH*')
    elif has_EH:
        # If only EH channels are present, select only EH
        # NTS: This may result in getting only vertical component data - EH? is used for PNSN analog stations
        # NTS: This may also be tricky for pulling full day-volumes because the sampling rate shifts for
        #      analog stations due to the remote digitization scheme used with analog stations
        sdata += st.select(id=f'{network}.{station}..EH*')
    else:
        print(f'No data available at {network}.{station} with HHZ, BHZ or EHZ channels at {time_pick}. Skipping.')
        continue

    # Resample
    sdata.resample(new_sampling_rate)
        
    # Apply highpass filter
    sdata.detrend(type='demean')
    sdata.taper(max_percentage=0.5)
    sdata.filter(type='highpass', freq=freq_highpass)
    # tr.filter(type='bandpass', freqmin=freqmin, freqmax=freqmax)

    max_amp = 0
    for tr in sdata:
        max_amp = max(max_amp, abs(tr.data).max())

    amplitudes.append(max_amp)

# Append to DataFrame
subset.loc[:,"Amplitude"] = amplitudes

PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09


PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09


In [6]:
# Print the subset of the data frame with the new "Amplitude" column
subset

,Unnamed: 0,time_pick,station,phase,timeres,idx,arid,latitude,longitude,depth,Detection Value,time,RMS Residual (s),Num. P,Num. S,picks,slatitude,slongitude,selevation,Amplitude
153817,153817,2011-04-16T03:50:43.228400Z,PB.B006,S,0.104,10894,153817,48.45886,-123.22030,20.6905,0.710,2011-04-16 03:50:28.547000+00:00,0.263,5,7,12,48.058800,-123.500801,302.0,8.009905
244439,244439,2011-12-20T16:07:51.797101Z,7D.J68A,P,0.162,17397,244439,43.83597,-129.03227,-1.0600,0.713,2011-12-20 16:06:43.285000+00:00,1.465,38,18,56,48.480999,-127.829201,-2590.0,393.388867
338682,338682,2012-07-01T12:51:08.738400Z,PB.B943,P,0.330,23010,338682,47.76690,-122.35029,19.5385,0.818,2012-07-01 12:51:00.754000+00:00,0.754,20,21,41,47.813202,-122.911301,84.2,7.775698
936755,936755,2015-03-27T21:27:52.968600Z,7D.FS10D,P,0.159,59912,936755,40.94625,-124.84570,20.5215,0.816,2015-03-27 21:27:42.856000+00:00,0.923,9,20,29,40.432800,-124.694000,-1153.8,42.333071
951058,951058,2015-04-15T07:43:32.452800Z,7D.G18D,P,-0.114,60738,951058,40.31837,-124.44135,17.9215,1.077,2015-04-15 07:43:15.290000+00:00,0.975,38,31,69,41.304699,-125.255798,-3130.0,18.008545
345552,345552,2012-07-11T11:27:13.178400Z,PB.B932,P,0.021,23520,345552,40.22253,-124.10574,10.9595,0.817,2012-07-11 11:27:10.741000+00:00,0.387,11,11,22,40.275002,-124.221199,52.0,4.453184
88023,88023,2010-09-17T07:54:49.698400Z,PB.B941,S,0.597,6223,88023,46.51102,-121.38421,2.0725,0.695,2010-09-17 07:54:24.736000+00:00,0.343,15,16,31,46.986801,-122.219002,151.0,9.234193
43966,43966,2010-05-02T18:17:42.458400Z,PB.B203,S,0.176,3224,43966,46.50157,-122.41299,18.4120,0.846,2010-05-02 18:17:30.386000+00:00,1.062,34,33,67,46.168999,-122.333664,814.4,9.023050
239972,239972,2011-12-10T12:40:25.298400Z,PB.B932,P,0.674,17089,239972,41.08989,-125.16140,16.8625,0.762,2011-12-10 12:40:08.233000+00:00,0.811,18,8,26,40.275002,-124.221199,52.0,4.970561
89204,89204,2010-09-20T12:02:25.110000Z,CN.GOBB,S,0.337,6301,89204,49.66736,-123.76562,1.5770,0.770,2010-09-20 12:02:01.842000+00:00,0.814,12,11,23,48.949300,-123.510500,173.0,24.237677
